# Phase 2 Step 2 v5 — Real CADD Scores (Batch Fetch)

**What this does:** Fetches real CADD phred scores from MyVariant.info in batches of 1000

**Runtime:** ~5-10 minutes total

**Output:** Same `clinvar_training_data_v2.csv` — Steps 4-8 need zero changes

**Run cells one by one**

In [ ]:
!pip install pandas numpy requests tqdm -q
import pandas as pd
import numpy as np
import requests
import time
import json
import os
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PATH = '/content/drive/My Drive/clinvar_training_data_v2.csv'
df = pd.read_csv(PATH)
print(f'Loaded: {df.shape}')
print(f'Labels: {df["label"].value_counts().to_dict()}')
print(f'Columns: {list(df.columns)}')
print(f'Chrom sample: {df["chrom"].sample(3).tolist()}')
print(f'Pos sample  : {df["pos"].sample(3).tolist()}')
print(f'Unique CADD : {df["cadd_phred"].nunique()} (simulated if <6000)')

In [ ]:
# Batch fetch function — 1000 positions per API call
def fetch_cadd_batch(positions):
    '''
    positions: list of (chrom, pos) tuples — chrom as string without chr prefix
    returns: dict {chrom:pos -> cadd_phred}
    '''
    results = {}
    try:
        queries = [f'chrom:{c} AND vcf.position:{p}' for c,p in positions]
        payload = {'q': queries, 'fields': 'cadd.phred', 'size': 1}
        resp = requests.post(
            'https://myvariant.info/v1/query',
            json=payload,
            timeout=60
        ).json()
        for i, item in enumerate(resp):
            chrom, pos = positions[i]
            key = f'{chrom}:{pos}'
            score = None
            if isinstance(item, dict):
                hits = item.get('hits', [])
                if hits:
                    cadd = hits[0].get('cadd', {})
                    if isinstance(cadd, list):
                        scores = [c.get('phred') for c in cadd if 'phred' in c]
                        score = max(scores) if scores else None
                    elif isinstance(cadd, dict):
                        score = cadd.get('phred')
            results[key] = float(score) if score is not None else None
    except Exception as e:
        for chrom, pos in positions:
            results[f'{chrom}:{pos}'] = None
    return results

# Test with 2 known variants
test = fetch_cadd_batch([('17', 41246590), ('13', 32906729)])
print(f'Batch test: {test}')
if any(v is not None for v in test.values()):
    print('API working')
else:
    print('API failed — do not proceed')

In [ ]:
# Build list of unique positions to fetch
# Strip chr prefix to match API format
CHECKPOINT = '/content/drive/My Drive/cadd_scores_checkpoint.json'

cadd_cache = {}
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f:
        cadd_cache = json.load(f)
    print(f'Loaded checkpoint: {len(cadd_cache):,} already fetched')
else:
    print('No checkpoint found - starting fresh')

# Build positions list — strip chr prefix
unique_pos = df[['chrom','pos']].drop_duplicates()
all_positions = []
for _, r in unique_pos.iterrows():
    chrom_clean = str(r.chrom).replace('chr','')
    pos_int = int(r.pos)
    key = f'{chrom_clean}:{pos_int}'
    if key not in cadd_cache:
        all_positions.append((chrom_clean, pos_int))

print(f'Total unique positions : {len(unique_pos):,}')
print(f'Already fetched        : {len(cadd_cache):,}')
print(f'Remaining              : {len(all_positions):,}')
print(f'Batches of 1000        : {len(all_positions)//1000 + 1}')
print(f'Estimated time         : {(len(all_positions)//1000 + 1) * 2:.0f} min')

In [ ]:
# Main batch fetch — ~5-10 minutes for 77K positions
BATCH_SIZE = 1000
batches = [all_positions[i:i+BATCH_SIZE]
           for i in range(0, len(all_positions), BATCH_SIZE)]

print(f'Running {len(batches)} batches...')

for i, batch in enumerate(tqdm(batches, desc='Fetching CADD')):
    results = fetch_cadd_batch(batch)
    cadd_cache.update(results)
    time.sleep(1)  # be polite to API

    # Save checkpoint every 10 batches
    if (i + 1) % 10 == 0:
        with open(CHECKPOINT, 'w') as f:
            json.dump(cadd_cache, f)
        tqdm.write(f'Checkpoint saved: {len(cadd_cache):,}')

# Final save
with open(CHECKPOINT, 'w') as f:
    json.dump(cadd_cache, f)

success = sum(1 for v in cadd_cache.values() if v is not None)
print(f'Done!')
print(f'Total fetched  : {len(cadd_cache):,}')
print(f'Successful     : {success:,} ({success/len(cadd_cache)*100:.1f}%)')
print(f'Not found/None : {len(cadd_cache)-success:,}')

In [ ]:
# Map real CADD scores back to training dataframe
def get_cadd(row):
    chrom_clean = str(row.chrom).replace('chr','')
    key = f'{chrom_clean}:{int(row.pos)}'
    return cadd_cache.get(key, None)

df['cadd_real'] = df.apply(get_cadd, axis=1)

has_real = df['cadd_real'].notna().sum()
missing  = df['cadd_real'].isna().sum()
print(f'Real CADD scores fetched: {has_real:,}/{len(df):,} ({has_real/len(df)*100:.1f}%)')
print(f'Missing (will fill)     : {missing:,}')

# Fill missing with label-specific median — NOT random normal
path_med = df[df['label']==1]['cadd_real'].median()
ben_med  = df[df['label']==0]['cadd_real'].median()
print(f'Pathogenic median CADD : {path_med:.2f}')
print(f'Benign median CADD     : {ben_med:.2f}')

df.loc[(df['label']==1) & df['cadd_real'].isna(), 'cadd_real'] = path_med
df.loc[(df['label']==0) & df['cadd_real'].isna(), 'cadd_real'] = ben_med

# Replace simulated scores with real ones
df['cadd_phred'] = df['cadd_real'].round(3)

print(f'Unique CADD values: {df["cadd_phred"].nunique():,}')
print('(Should be >>10,000 confirming real scores)')
print()
print('Pathogenic CADD stats:')
print(df[df['label']==1]['cadd_phred'].describe().round(3))
print()
print('Benign CADD stats:')
print(df[df['label']==0]['cadd_phred'].describe().round(3))

In [ ]:
import matplotlib.pyplot as plt
import os
os.makedirs('figures_step2', exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
path = df[df['label']==1]['cadd_phred']
ben  = df[df['label']==0]['cadd_phred']
ax.hist(path, bins=60, alpha=0.6, color='#C0392B',
        label=f'Pathogenic n={len(path):,} mean={path.mean():.1f}', density=True)
ax.hist(ben,  bins=60, alpha=0.6, color='#2E5FA3',
        label=f'Benign n={len(ben):,} mean={ben.mean():.1f}', density=True)
ax.set_xlabel('CADD Phred Score (REAL — fetched from MyVariant.info)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('ClinVar Training Data — Real CADD Score Distributions\n'
             'Pathogenic vs Benign (genuine biological signal)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step2/real_cadd_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved')

In [ ]:
import shutil

# Keep same columns as before — zero changes needed in Steps 4-8
out_cols = [c for c in ['chrom','pos','ref','alt','gene','variant_type',
            'impact','cadd_phred','gerp_rs','sift_score',
            'polyphen_score','spliceai_score','label']
            if c in df.columns]
training = df[out_cols].copy()

local_path = 'clinvar_training_data_v2.csv'
drive_path = '/content/drive/My Drive/clinvar_training_data_v2.csv'

training.to_csv(local_path, index=False)
shutil.copy(local_path, drive_path)

size_kb = os.path.getsize(drive_path) / 1e3
print(f'Saved to Drive: {drive_path}')
print(f'Size: {size_kb:.0f} KB | Rows: {len(training):,}')
print(f'Unique CADD values: {training["cadd_phred"].nunique():,}')
print(f'Pathogenic mean CADD: {training[training["label"]==1]["cadd_phred"].mean():.3f}')
print(f'Benign mean CADD    : {training[training["label"]==0]["cadd_phred"].mean():.3f}')
print()
print('Steps 4, 5, 6, 7, 8 need ZERO changes')
print('Re-run Step 4 next')